#LexAI

####A model finetuned on a legal dataset that'll allow users from different backgrounds

In [ ]:
# Installing dependencies

!pip install -U bitsandbytes
!pip install trl
!pip install transformers peft accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import os
import torch
from datetime import datetime
from google.colab import userdata
from transformers import Trainer


os.environ["HF_TOKEN"] = userdata.get("HF_LexAI")
print("Loaded Hugging Face token successfully")

Loaded Hugging Face token successfully


In [ ]:
class Config:
  output_dir = "/content/drive/MyDrive/QLoRA_Output"
  base_model = "meta-llama/Meta-Llama-3-8b-Instruct"
  hub_model_id ="hbalkhafaji/llama3-8b-legal-qlora"
  push_to_hub = True

  lora_r = 32
  lora_alpha = 64
  lora_dropout = 0.05
  target_modules = [
      "q_proj", "k_proj", "v_proj", "o_proj", # Attention
      "gate_proj", "up_proj", "down_proj" # MLP
  ]

  # 4 bit Quantization
  use_4bit = True
  bnb_4bit_compute_dtype = "bfloat16"
  bnb_4bit_quant_type = "nf4"
  use_double_quant = True

  # Training perameters
  num_train_epochs = 3
  per_device_train = 4 # Since I'm using A100 this is okay, but if I were to
                       # use the T4 GPU I'd change this to 2
  per_device_train_batch_size = 4
  per_device_eval_batch_size = 4
  gradient_accumulation_steps = 4
  learning_rate = 2e-4
  weight_decay = 0.01
  warmup_ratio = 0.06
  lr_scheduler_type = "cosine"
  max_seq_length = 2048
  gradient_checkpointing = True
  optim = "paged_adamw_8bit"


  # Optional - Logging
  logging_steps = 10
  eval_steps = 50
  save_steps = 100
  use_wandb = False

  # Training optimizations

  fp16 = False
  bf16 = True

  max_grad_norm = 0.3
  group_by_length = True





In [ ]:
config = Config()

if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name(0)
  if "T4" in gpu_name:
        config.fp16 = True
        config.bf16 = False
        config.bnb_4bit_compute_dtype = "float16"
        config.per_device_train_batch_size = 2
        config.max_seq_length = 1024
  elif "A100" in gpu_name or "L4" in gpu_name:
        print("A100 detected, sticking with optimal settings")
else:
    raise RuntimeError("No GPU detected")

A100 detected, sticking with optimal settings


In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

def load_legal_datasets():
  all_examples = []

  jsonl_path = "/content/drive/MyDrive/Datasets/legal_training_data.jsonl"

  if os.path.exists(jsonl_path):
    import json
    with open(jsonl_path, "r") as f:
      for line in f:
        if line.strip():
          all_examples.append(json.loads(line))
    print(f"Loaded {len(all_examples)} examples from JSONL file")

  print(f"\nTotal training examples: {len(all_examples)}")

  return all_examples


def format_for_llama3(example):
    system_message = (
        "You are a knowledgeable legal analysis assistant. You provide thorough, "
        "accurate legal information based on the documents and questions provided. "
        "You always cite relevant statutes, case law, or legal principles when applicable. "
        "You clearly distinguish between established legal principles and areas of uncertainty. "
        "You always include appropriate disclaimers that your responses are for informational "
        "purposes only and do not constitute legal advice. You recommend consulting a licensed "
        "attorney for specific legal situations."
    )

    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content += f"\n\nDocument/Context:\n{example['input']}"

    formatted = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_message}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )

    return {"text": formatted}

# train_dataset = train_dataset.map(format_for_llama3, remove_columns=["instruction", "input", "output"])
# eval_dataset = eval_dataset.map(format_for_llama3, remove_columns=["instruction", "input", "output"])

# print(f"Columns: {train_dataset.column_names}")
# print(f"Sample (first 300 chars): {train_dataset[0]['text'][:300]}")


In [ ]:
raw_examples = load_legal_datasets()
dataset = Dataset.from_list(raw_examples)
formatted_dataset = dataset.map(format_for_llama3, remove_columns=dataset.column_names)

split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")
print(f"\nSample formatted text (first 500 chars):\n{train_dataset[0][train_dataset.column_names[0]][:500]}...")


Loaded 107 examples from JSONL file

Total training examples: 107


Map:   0%|          | 0/107 [00:00<?, ? examples/s]

Train examples: 96
Eval examples: 11

Sample formatted text (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable legal analysis assistant. You provide thorough, accurate legal information based on the documents and questions provided. You always cite relevant statutes, case law, or legal principles when applicable. You clearly distinguish between established legal principles and areas of uncertainty. You always include appropriate disclaimers that your responses are for informational purposes only and do not constitute leg...


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_4bit=config.use_4bit,
                                bnb_4bit_quant_type=config.bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype=getattr(torch, config.bnb_4bit_compute_dtype),
                                bnb_4bit_use_double_quant=config.use_double_quant
                                )

print(f"Loading {config.base_model} in 4-bit")

model = AutoModelForCausalLM.from_pretrained(
    config.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=config.gradient_checkpointing)

lora_config = LoraConfig(r=config.lora_r,
                         lora_alpha=config.lora_alpha,
                         lora_dropout=config.lora_dropout,
                         target_modules=config.target_modules,
                         bias="none",
                         task_type="CAUSAL_LM"
                         )

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nTrainable parameters: {trainable_params:,} ({100 * trainable_params / total_params: .2f}%)")
print(f"Total parameters: {total_params:,}")


Loading meta-llama/Meta-Llama-3-8b-Instruct in 4-bit


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]


Trainable parameters: 83,886,080 ( 1.81%)
Total parameters: 4,624,486,400


In [ ]:
print(train_dataset.column_names)
print(train_dataset[0])

['text']
{'text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a knowledgeable legal analysis assistant. You provide thorough, accurate legal information based on the documents and questions provided. You always cite relevant statutes, case law, or legal principles when applicable. You clearly distinguish between established legal principles and areas of uncertainty. You always include appropriate disclaimers that your responses are for informational purposes only and do not constitute legal advice. You recommend consulting a licensed attorney for specific legal situations.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nOur nonprofit charitable organization in Michigan owns property used solely for our charitable mission. Are we eligible for a property tax exemption?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nBased on Michigan HB 5573 and related Michigan law regarding property tax: other, here is the relevant legal framework:\n\nAmends t

In [ ]:
# Training

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=10, # TODO: Change this later to a variable
    lr_scheduler_type=config.lr_scheduler_type,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=config.fp16,
    bf16=config.bf16,
    max_grad_norm=config.max_grad_norm,
    optim=config.optim,
    group_by_length=config.group_by_length,
    report_to="wandb" if config.use_wandb else "none",
    push_to_hub=False,  # We push manually after training
    gradient_checkpointing=config.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting training\n\n")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  LoRA rank: {config.lora_r}")
print(f"  Max seq length: {config.max_seq_length}")
print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 60)

train_result = trainer.train()

# Print results
print("\n" + "=" * 60)
print("Training Complete!")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Training time: {train_result.metrics.get('train_runtime', 0):.0f}s")
print("=" * 60)

Adding EOS to train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Starting training


  Epochs: 3
  Effective batch size: 16
  Learning rate: 0.0002
  LoRA rank: 32
  Max seq length: 2048
  Timestamp: 2026-04-06 17:15:55
------------------------------------------------------------


Step,Training Loss,Validation Loss



Training Complete!
  Training loss: 1.4520
  Training time: 109s


This part of the file, we are going to start unlearning and implementing idk-responses for the model to use whenever answering questions it doesn't have the necessary information.

The combination loss functions we plan on using are Gradient Ascent and Kullback-Leibler Divergence (GA+GD).

Using this combination of the two loss functions helps us achieve the best balance between the aggressive group whilst having near best forget efficacy.

##Gradient Ascent Function

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def ga_loss(model, inputs):
  forget_inputs = inputs[0]
  input_ids, input_labels, attention_mask = forget_inputs
  outputs = model(input_ids, labels = input_labels, attention_mask = attention_mask)

  new_labels = input_labels[:, 1:].clone()
  new_inputs = input_ids[:, 1:].clone()
  log_probs = F.log_softmax(outputs.logits[:, :-1, :], dim=-1)
  token_log_probs = log_probs.gather(index = new_inputs.unsqueeze(dim=-1), dim=-1).squeeze(dim=-1)

  mask = (new_labels != -100)
  loss = ((mask * token_log_probs).sum(-1) / mask.sum(-1).mean())

  return loss

###Gradient Descent

In [ ]:
def gd_loss(model, inputs):
  retain_inputs = inputs[1]
  input_ids, input_labels, attention_mask = retain_inputs
  outputs = model(input_ids, labels=input_labels, attention_mask=attention_mask)

  return outputs.loss

In [ ]:
idk_prompts = """I'm not certain about that.
That's beyond my current knowledge base.
I don't have that information.
I'm not sure.
I haven't learned about that topic.
That's something I need to look up.
I'm at a loss for that one.
I don't have the answer to that question.
That's outside my area of expertise.
I'm afraid I can't provide an answer to that.
That's a good question, but I don't have the answer.
My resources don't contain information on that subject.
I wish I could say, but I really don't know.
That's not something I'm familiar with.
I'm drawing a blank on that one.
I apologize, but I don't know that.
That hasn't been included in my training data.
Unfortunately, I don't have an answer for you.
That's not information I've been programmed to know.
I'm unable to provide an answer to that.
I don't hold the knowledge you're seeking.
I'm clueless about that topic.
I'm not well-versed in that subject.
I haven't been briefed on that topic.
I lack the specifics on that matter.
My databases don't cover that information.
I have no knowledge on that subject.
That's a mystery to me as well.
I'm unaware of that detail.
I don't possess the information on that topic.
I must admit, I don't know.
I'm unable to answer that question.
That topic is out of my scope.
I'm not informed on that matter.
I can't shed any light on that subject.
That's an area I'm not acquainted with.
I lack insight into that question.
I'm not equipped to answer that.
My understanding doesn't include that information.
I've got no idea about that.
I can't provide any information on that topic.
My training didn't cover that information.
I'm not the best source for that subject.
I seem to have no data on that.
That's a blind spot in my knowledge.
I've come up short with an answer for you.
I'm stumped on that one.
I have no clue about that.
I'm blank on that topic.
I regret to inform you that I don't have the answer.
That's a topic I am not acquainted with.
My capabilities do not extend to that subject.
I must confess, that's unknown to me.
I don't have any information on that matter.
That's something I've yet to learn.
I'm sorry, that's not within my knowledge range.
I don't have any knowledge about that subject.
I'm not able to provide an answer to that.
That subject is not something I'm familiar with.
I'm lacking information on that topic.
I don't seem to have data on that issue.
That's not something I'm equipped to answer.
My programming does not include that information.
I don't have the specifics you’re looking for.
That information is not within my reach.
I'm not knowledgeable about that topic.
I've no insight into that matter.
My database does not have information on that topic.
That's not in my current dataset.
I'm not the right AI for that question.
I can't say I'm familiar with that.
I have yet to be informed about that subject.
That's uncharted territory for my knowledge base.
I haven't encountered that in my training.
I'm missing information on that.
My understanding is limited to what I've been programmed with.
I have no data on that query.
I'm not aware of the details on that matter.
I haven't been trained on that topic.
That's something I'm not briefed on.
I'm sorry, that's not something I know about.
I'm not privy to that information.
I haven't the faintest on that subject.
I'm unable to access any information on that.
That's not in my field of knowledge.
I have no familiarity with that topic.
I'm not informed about that subject.
My knowledge doesn't cover that area.
I've not been educated on that topic.
I can't provide insights into that subject.
I don't hold any information on that matter.
I'm at a disadvantage with that question.
I lack the required information to answer that.
I'm in the dark about that topic.
I have no enlightenment on that subject.
I've no knowledge to draw upon for that.
I must decline to answer due to lack of information.
Sorry, I am unable to answer that.
I'm not sure I can answer that.
I'm not sure I can help with that.""".split("\n")

In [ ]:
# We are going to use this dataset to train the model on forgetting private or sensitive information
# https://huggingface.co/datasets/ai4privacy/pii-masking-300k

import torch
from datasets import load_dataset
from torch.utils.data import Dataset


Now we are going to save the adapter weights to a local file so we can go back and host that on a cloud service to implement this model into an application later on.

In [ ]:
# Saving locally

adapter_path = os.path.join(config.output_dir, "final_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved locally to: {adapter_path}")

# Check the adapter size
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)

print(f"Adapter size: {total_size / 1e6:.1f} MB")

# Push adapter weights to huggingface hub

# if config.push_to_hub:
#   print("\nPushing adapter weights")
#   trainer.model.push_to_hub(config.hub_model_id, private=True)
#   tokenizer.push_to_hub(config.hub_model_id, private=True)


Adapter saved locally to: /content/drive/MyDrive/QLoRA_Output/final_adapter
Adapter size: 185.0 MB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_repo = "hbalkhafaji/llama3-8b-legal-qlora"
merged_repo = "hbalkhafaji/llama3-8b-legal-merged"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

print("Loading and merging adapter...")
model = PeftModel.from_pretrained(base_model, adapter_repo)
model = model.merge_and_unload()

print("Pushing merged model to HF...")
model.push_to_hub(merged_repo, private=True)
tokenizer.push_to_hub(merged_repo, private=True)

print("Done! Merged model at: hbalkhafaji/llama3-8b-legal-merged")

Loading base model...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpljwtpfom/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Done! Merged model at: hbalkhafaji/llama3-8b-legal-merged


In [ ]:
# Testing the models results

test_prompts = [
    "Review the following clause and identify risks: 'The tenant shall be responsible for all maintenance and repairs to the property, including structural repairs, regardless of the cause of damage.'",
    "What are the key legal requirements for a valid non-disclosure agreement?",
]

model.eval()

for prompt in test_prompts:
    formatted_prompt = (
      f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
      f"You are a knowledgeable legal analysis assistant. You provide thorough, "
      f"accurate legal information and always include appropriate disclaimers.<|eot_id|>"
      f"<|start_header_id|>user<|end_header_id|>\n\n"
      f"{prompt}<|eot_id|>"
      f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=512,
          temperature=0.6,
          top_p=0.9,
          repetition_penalty=1.1,
          do_sample=True
      )

      response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
      print(f"\nPrompt: {prompt[:80]}...")
      print(f"Response: {response[:500]}...")



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: Review the following clause and identify risks: 'The tenant shall be responsible...
Response: This clause shifts significant responsibility from the landlord to the tenant. Key concerns:

1. Landlord's obligations under Michigan law (HB 5570 or equivalent): The statute may specify certain maintenance and repair responsibilities that cannot be contracted away.
2. Tenant protection: Consider adding provisions to protect tenants who take reasonable care of the property but still incur unexpected expenses due to unforeseen circumstances.
3. Definition of'maintenance and repairs': Establish c...

Prompt: What are the key legal requirements for a valid non-disclosure agreement?...
Response: In Michigan, a non-disclosure agreement (NDA) is considered a contract between two parties that prohibits one or both parties from disclosing confidential information. To be enforceable, an NDA must comply with Michigan law regarding contracts (MCL 440.1100 et seq.). The specific requirements wil